# Example 6.5.1 / 1: Explicit Component Lists

Companion notebook for **6.5.1 Named Problems and Environments** in the AMPL lab Documentation, section "Explicit Component Lists". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Explicit Component Lists

In the Python tabs, `ampl` has already read `model.mod`. amplpy has no method for the `problem`
command, so it is sent with `ampl.eval()`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 0;
var y >= 0;
minimize ObjA: x + 2*y;
minimize ObjB: 3*x + y;
subject to Link: x + y >= 10;

problem A: x, y, ObjA, Link;
problem B: x, y, ObjB, Link;

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# run.py
for name in ["A", "B"]:
    ampl.eval(f"problem {name};")
    ampl.option["solver_msg"] = 0
    ampl.solve(solver="highs")
    print(name, ampl.var["x"].value(), ampl.var["y"].value())

A 10.0 0.0
B 0.0 10.0


### Pure AMPL: model.run

The same in pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
model model.mod;

In [6]:
%%ampl_eval
# model.run
problem A;
option solver highs, solver_msg 0;
solve;
display x, y;
problem B;
option solver highs, solver_msg 0;
solve;
display x, y;

x = 10
y = 0

x = 0
y = 10

